In [1]:
import pandas as pd


In [7]:
from selenium.common.exceptions import NoSuchElementException

In [113]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time


options = Options()

driver = webdriver.Chrome(options=options)
driver.maximize_window()


In [115]:
url = 'https://www.topuniversities.com/world-university-rankings'
driver.get(url)
time.sleep(4)

## IMPORTANT Note : once above page gets loaded, switch to table view there manually and then run below codes.

In [119]:
import re
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)
is_number = lambda s: re.match(r'^\d+(\.\d+)?$', s) is not None

all_rows = []
page_num = 1

while True:
    
    wait.until(EC.presence_of_element_located((By.ID, "ranking-data-load")))

    
    table = driver.find_elements(By.XPATH, '//*[@id="ranking-data-load"]')

    
    texts = []
    for el in table:
        for line in el.text.split("\n"):
            line = line.strip()
            if line:
                texts.append(line)

    i = 0
    page_rows = []
    while i < len(texts):
        if i + 3 > len(texts):
            print(f"Page {page_num}: leftover incomplete data at end:", texts[i:])
            break
        rank = texts[i]; i += 1
        university = texts[i]; i += 1
        location = texts[i]; i += 1
        
        while i < len(texts) and not is_number(texts[i]):
            i += 1
        if i + 4 > len(texts):
            print(f"Page {page_num}: incomplete scores for {university}, remaining: {texts[i:]}")
            break
        scores = texts[i:i+4]
        i += 4
        page_rows.append([rank, university, location, *scores])

    print(f"Page {page_num}: parsed {len(page_rows)} rows out of {len(texts)} tokens")
    all_rows.extend(page_rows)

    
    try:
        next_btn = driver.find_element(By.XPATH, '//a[contains(@class, "next")]')
        parent_li = next_btn.find_element(By.XPATH, '..')

        if 'disabled' in parent_li.get_attribute('class'):
            print("Reached last page.")
            break

        next_btn.click()
        time.sleep(2)
        page_num += 1

    except Exception as e:
        print("No next button found — stopping.", e)
        break


df = pd.DataFrame(all_rows, columns=[
    "Rank", "University", "Location",
    "Score1", "Score2", "Score3", "Score4"
])
print(f"\nTotal parsed: {len(df)} rows across {page_num} pages")


Page 1: parsed 30 rows out of 220 tokens
Page 2: parsed 30 rows out of 223 tokens
Page 3: parsed 30 rows out of 223 tokens
Page 4: parsed 30 rows out of 216 tokens
Page 5: parsed 30 rows out of 231 tokens
Page 6: parsed 30 rows out of 222 tokens
Page 7: parsed 30 rows out of 230 tokens
Page 8: parsed 30 rows out of 218 tokens
Page 9: parsed 30 rows out of 219 tokens
Page 10: parsed 30 rows out of 218 tokens
Page 11: parsed 30 rows out of 223 tokens
Page 12: parsed 30 rows out of 224 tokens
Page 13: parsed 30 rows out of 216 tokens
Page 14: parsed 30 rows out of 227 tokens
Page 15: parsed 30 rows out of 227 tokens
Page 16: parsed 30 rows out of 215 tokens
Page 17: parsed 30 rows out of 213 tokens
Page 18: parsed 30 rows out of 219 tokens
Page 19: parsed 30 rows out of 221 tokens
Page 20: parsed 30 rows out of 218 tokens
Page 21: incomplete scores for Santiago de Compostela, Spain, remaining: ['35.2', '15.5', '17.6']
Page 21: parsed 29 rows out of 219 tokens
Page 22: parsed 30 rows out o

,Rank,University,Location,Score1,Score2,Score3,Score4
0,1,Massachusetts Institute of Technology (MIT),"Cambridge, United States",100,100,100,100
1,=2,Imperial College London,"London, United Kingdom",93.8,99.6,98.9,100
2,=2,Stanford University,"Stanford, United States",98.8,100,100,100
3,4,University of Oxford,"Oxford, United Kingdom",89,100,100,100
4,5,Harvard University,"Cambridge, United States",100,100,97.4,100
...,...,...,...,...,...,...,...
1498,1401+,University of Sri Jayewardenapura,"Nugegoda, Sri Lanka",7.4,7.5,4.6,11.6
1499,1401+,University of Tyumen,"Tyumen, Russia",2.4,1.9,41.5,5.7
1500,1401+,Walailak University,"Thasala, Thailand",5.9,9.5,12.8,3.3
1501,1401+,Yessenov University,"Aktau, Kazakhstan",1.3,12.7,14.9,7.3


In [121]:
df

,Rank,University,Location,Score1,Score2,Score3,Score4
0,1,Massachusetts Institute of Technology (MIT),"Cambridge, United States",100,100,100,100
1,=2,Imperial College London,"London, United Kingdom",93.8,99.6,98.9,100
2,=2,Stanford University,"Stanford, United States",98.8,100,100,100
3,4,University of Oxford,"Oxford, United Kingdom",89,100,100,100
4,5,Harvard University,"Cambridge, United States",100,100,97.4,100
...,...,...,...,...,...,...,...
1498,1401+,University of Sri Jayewardenapura,"Nugegoda, Sri Lanka",7.4,7.5,4.6,11.6
1499,1401+,University of Tyumen,"Tyumen, Russia",2.4,1.9,41.5,5.7
1500,1401+,Walailak University,"Thasala, Thailand",5.9,9.5,12.8,3.3
1501,1401+,Yessenov University,"Aktau, Kazakhstan",1.3,12.7,14.9,7.3


In [125]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Rank        1503 non-null   object
 1   University  1503 non-null   object
 2   Location    1503 non-null   object
 3   Score1      1503 non-null   object
 4   Score2      1503 non-null   object
 5   Score3      1503 non-null   object
 6   Score4      1503 non-null   object
dtypes: object(7)
memory usage: 82.3+ KB


In [127]:
df.to_clipboard(excel=True)

In [ ]:
driver.quit()